# Lineage — Unity Catalog Lineage Showcase

**Doel:** Tonen hoe Unity Catalog automatisch herkomst bijhoudt voor elke tabel in het
medallion-lakehouse (Bronze → Silver → Gold) — zowel visueel via de Catalog Explorer
als programmatisch via de `system.lineage`-tabellen.

## Wat is Unity Catalog Lineage?

Unity Catalog legt automatisch **tabel-level** en **kolom-level** herkomst vast:

| Niveau | Wat het toont |
|---|---|
| **Tabel-level** | Welke brontabellen of bestanden zijn gelezen om deze tabel te vullen? |
| **Kolom-level** | Welke bronkolom levert de waarde voor elke doelkolom? |

Herkomst wordt automatisch vastgelegd — geen extra instrumentatie, annotaties of
ETL-aanpassingen nodig. Elke notebook-run, DLT-pipeline-refresh en SQL-query die
data leest of schrijft naar een Unity Catalog-tabel registreert de lineage.

## Inhoud van dit notebook

| Sectie | Beschrijving |
|---|---|
| 1 | Preflight-check — `system.lineage.table_lineage` bereikbaar? |
| 2 | Walkthrough: Bronze `STAGING_AZURESTORAGE.order_header` — upstream parquet-volume |
| 3 | Walkthrough: Silver `INTEGRATION.sales_line` — upstream Bronze-tabellen via CDF + FLOW AUTO CDC |
| 4 | Walkthrough: Gold `DATAMART.daily_sales_by_truck` — upstream Silver |
| 5 | Walkthrough: Gold `DATAMART.sales_lines_wide` — volledige keten Bronze → Silver → Gold |
| 6 | Programmatische lineage-query — impact-analyse en audit via `system.lineage` |
| 7 | Waarde van lineage voor impact-analyse, debugging en compliance |

## Medallion-architectuur — lineage op één oogopslag

```
/Volumes/.../parquet/ORDER_HEADER*.parquet  ──┐
                                               ├─→  STAGING_AZURESTORAGE.order_header  (Bronze)
/Volumes/.../parquet/ORDER_DETAIL*.parquet  ──┘         │
                                                         │  CDF + FLOW AUTO CDC (DLT)
                                                         ▼
                              INTEGRATION.order_header   ┐
                              INTEGRATION.order_detail   ┘  →  INTEGRATION.sales_line  (Silver MV)
                                        │                              │
                                        │                              │
                                        ▼                              ▼
                     DATAMART.daily_sales_by_truck          DATAMART.sales_lines_wide  (Gold MV)
```

> **Let op:** Dit notebook voert alleen lees-queries uit. Herdraaien is volledig
> non-destructief.

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Demo catalog")

catalog = dbutils.widgets.get("catalog")

# Schema-namen
bronze_schema  = "STAGING_AZURESTORAGE"
silver_schema  = "INTEGRATION"
gold_schema    = "DATAMART"

# Vier lineage-targets uit de issue-spec
bronze_order_header       = f"{catalog}.{bronze_schema}.order_header"
silver_sales_line         = f"{catalog}.{silver_schema}.sales_line"
gold_daily_sales_by_truck = f"{catalog}.{gold_schema}.daily_sales_by_truck"
gold_sales_lines_wide     = f"{catalog}.{gold_schema}.sales_lines_wide"

print(f"Catalog         : {catalog}")
print(f"Bronze target   : {bronze_order_header}")
print(f"Silver target   : {silver_sales_line}")
print(f"Gold target 1   : {gold_daily_sales_by_truck}")
print(f"Gold target 2   : {gold_sales_lines_wide}")
print(f"Lineage bron    : system.lineage.table_lineage")

## Sectie 1 — Preflight-check: `system.lineage.table_lineage` bereikbaar?

Unity Catalog Lineage is beschikbaar via het `system.lineage`-schema, mits:

1. **System Tables zijn ingeschakeld** — ga naar *Account Console → Metastore → System schemas*
   en schakel `lineage` in.
2. **DBR 13.3 LTS of hoger** — lineage-tracking vereist Unity Catalog-aware compute.
3. **Tabellen zijn al minstens één keer beschreven** — herkomst verschijnt pas nadat
   een pipeline of query de tabel heeft gelezen of beschreven.

De preflight-check detecteert ontbrekende System Tables en geeft een duidelijke
instructie — zodat de klant niet met een verwarrende SQL-fout wordt geconfronteerd.

In [ ]:
# Preflight-check: bestaat system.lineage.table_lineage?
try:
    spark.sql("DESCRIBE TABLE system.lineage.table_lineage").collect()
    print("OK — system.lineage.table_lineage is bereikbaar.")
except Exception as e:
    raise RuntimeError(
        "\n"
        "=========================================================\n"
        "FOUT: system.lineage.table_lineage is NIET beschikbaar.\n"
        "=========================================================\n"
        "\n"
        "De Unity Catalog Lineage System Tables zijn niet ingeschakeld voor dit metastore.\n"
        "\n"
        "Stappen om dit op te lossen:\n"
        "  1. Ga naar Account Console (https://accounts.azuredatabricks.net).\n"
        "  2. Navigeer naar: Catalog → <metastore-naam> → System schemas.\n"
        "  3. Schakel het schema 'lineage' in.\n"
        "  4. Wacht 10–30 minuten tot de eerste lineage-records verschijnen.\n"
        "  5. Voer dit notebook opnieuw uit.\n"
        "\n"
        "Documentatie: https://docs.databricks.com/en/admin/system-tables/lineage.html\n"
        f"\nOriginele fout: {e}"
    )

## Sectie 2 — Bronze: `STAGING_AZURESTORAGE.order_header`

### Wat laat dit zien?

`STAGING_AZURESTORAGE.order_header` is de eerste Delta-tabel in de medallion-stack.
Ze wordt gevuld door `staging/02_ingest_azurestorage.ipynb` via Auto Loader
(`cloudFiles`) of een batch-read, afhankelijk van de `load_type`-instelling in de
control table.

**Verwachte upstream-herkomst (UC Lineage viewer):**

```
/Volumes/demo/staging_azurestorage/parquet/ORDER_HEADER*.parquet
    └─→  STAGING_AZURESTORAGE.order_header
```

Unity Catalog registreert automatisch dat de tabel wordt gevuld vanuit bestanden
in het external volume — geen handmatige annotatie vereist.

### Stap-voor-stap: Catalog Explorer

1. Ga in de Databricks-sidebar naar **Catalog** (boekje-icoontje).
2. Navigeer naar `DEMO` → `STAGING_AZURESTORAGE` → `order_header`.
3. Klik op het tabblad **Lineage**.
4. Klik op **See lineage graph** of scroll naar het stroomdiagram.
5. Je ziet één upstream-node: het parquet-volume
   `/Volumes/demo/staging_azurestorage/parquet`.
6. Klik op de volume-node — de sidebar toont het bestandspad en de pipeline-run die
   het bestand heeft ingevoerd.

> **Demo talking point:** De bron-tracing werkt naar bestanden — niet alleen naar tabellen.
> Klanten kunnen zien welk parquet-bestand een specifieke rij in Bronze heeft veroorzaakt.

In [ ]:
# Programmatische lineage-check voor STAGING_AZURESTORAGE.order_header (upstream)
bronze_upstream_sql = f"""
SELECT
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM system.lineage.table_lineage
WHERE LOWER(target_table_full_name) = LOWER('{bronze_order_header}')
ORDER BY created_at DESC
LIMIT 20
"""

bronze_upstream_df = spark.sql(bronze_upstream_sql)
count = bronze_upstream_df.count()
print(f"Upstream-herkomst voor '{bronze_order_header}': {count} record(s)")
display(bronze_upstream_df)

## Sectie 3 — Silver: `INTEGRATION.sales_line`

### Wat laat dit zien?

`INTEGRATION.sales_line` is een **Materialised View** die wordt berekend door de
DLT Silver-pipeline (`integration/05_silver_dlt_pipeline.ipynb`). Ze joint
`INTEGRATION.order_header` en `INTEGRATION.order_detail` op `order_id`.

Beide Silver Streaming Tables worden op hun beurt gevuld via **Change Data Feed +
`FLOW AUTO CDC`** vanuit de Bronze-tabellen (zie ADR 0002).

**Verwachte upstream-herkomst (UC Lineage viewer):**

```
STAGING_AZURESTORAGE.order_header  ──┐
                                      ├─→  INTEGRATION.order_header  ──┐
STAGING_AZURESTORAGE.order_detail  ──┘                                  ├─→  INTEGRATION.sales_line
                                      ├─→  INTEGRATION.order_detail  ──┘
STAGING_AZURESTORAGE.order_detail  ──┘
```

UC Lineage toont de directe upstream (`order_header` en `order_detail` in Silver)
én, bij uitklapbaar niveau, de verder-upstream Bronze-tabellen.

### Stap-voor-stap: Catalog Explorer

1. Ga in de Databricks-sidebar naar **Catalog**.
2. Navigeer naar `DEMO` → `INTEGRATION` → `sales_line`.
3. Klik op het tabblad **Lineage**.
4. Je ziet twee directe upstream-nodes: `INTEGRATION.order_header` en
   `INTEGRATION.order_detail`.
5. Klik op een van de nodes om verder omhoog te lopen in de keten:
   `order_header` (Silver) → `order_header` (Bronze) → parquet-volume.
6. Bekijk het tabblad **Column lineage** — klik op een kolom als `order_total`
   om te zien via welke bron-kolom de waarde binnenkomt.

> **Demo talking point:** De DLT-pipeline schrijft de lineage automatisch — ook voor
> `FLOW AUTO CDC`-operaties (CDF MERGE). Klanten hoeven niets te configureren.

In [ ]:
# Programmatische lineage-check voor INTEGRATION.sales_line (upstream)
silver_upstream_sql = f"""
SELECT
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM system.lineage.table_lineage
WHERE LOWER(target_table_full_name) = LOWER('{silver_sales_line}')
ORDER BY created_at DESC
LIMIT 20
"""

silver_upstream_df = spark.sql(silver_upstream_sql)
count = silver_upstream_df.count()
print(f"Upstream-herkomst voor '{silver_sales_line}': {count} record(s)")
display(silver_upstream_df)

## Sectie 4 — Gold: `DATAMART.daily_sales_by_truck`

### Wat laat dit zien?

`DATAMART.daily_sales_by_truck` is een Gold **Materialised View** die dagelijkse
revenue per truck berekent vanuit `INTEGRATION.order_header`. Ze wordt beheerd
door de DLT Gold-pipeline (`datamart/*.sql`).

**Verwachte upstream-herkomst (UC Lineage viewer):**

```
STAGING_AZURESTORAGE.order_header
    └─→  INTEGRATION.order_header
             └─→  DATAMART.daily_sales_by_truck
```

De twee-lagen-keten (Bronze → Silver → Gold) is volledig traceerbaar in de Lineage viewer.

### Stap-voor-stap: Catalog Explorer

1. Ga in de Databricks-sidebar naar **Catalog**.
2. Navigeer naar `DEMO` → `DATAMART` → `daily_sales_by_truck`.
3. Klik op het tabblad **Lineage**.
4. Je ziet één directe upstream-node: `INTEGRATION.order_header`.
5. Klik op `INTEGRATION.order_header` — de sidebar toont het schema en de pipeline
   die de Silver-tabel vult.
6. Klik in het stroomdiagram op de pijl **verder omhoog** om de Bronze-bron te zien:
   `STAGING_AZURESTORAGE.order_header` → parquet-volume.

> **Demo talking point:** Zelfs geaggregeerde Gold-tabellen hebben een traceerbare keten
> terug naar de ruwe bronbestanden. Als een metric-afwijking wordt gesignaleerd op een
> dashboard, kan een engineer binnen seconden de bronlaag opsporen.

In [ ]:
# Programmatische lineage-check voor DATAMART.daily_sales_by_truck (upstream)
gold_truck_upstream_sql = f"""
SELECT
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM system.lineage.table_lineage
WHERE LOWER(target_table_full_name) = LOWER('{gold_daily_sales_by_truck}')
ORDER BY created_at DESC
LIMIT 20
"""

gold_truck_upstream_df = spark.sql(gold_truck_upstream_sql)
count = gold_truck_upstream_df.count()
print(f"Upstream-herkomst voor '{gold_daily_sales_by_truck}': {count} record(s)")
display(gold_truck_upstream_df)

## Sectie 5 — Gold: `DATAMART.sales_lines_wide`

### Wat laat dit zien?

`DATAMART.sales_lines_wide` is de AI/BI-vriendelijke brede tabel op regel-grain
met **Liquid Clustering** op `(truck_id, location_id, order_date, order_currency)`.
Ze leest van `INTEGRATION.sales_line` — zelf een join-MV van twee Silver-tabellen
die elk vanuit Bronze worden gevuld.

Dit is de **langste lineage-keten** in de demo:

```
ORDER_HEADER*.parquet  ──┐
                          ├─→  STAGING_AZURESTORAGE.order_header  ──┐
                          │                                          ├─→  INTEGRATION.order_header  ──┐
ORDER_DETAIL*.parquet  ──┤                                          │                                  ├─→  INTEGRATION.sales_line
                          │                                          │                                  │       └─→  DATAMART.sales_lines_wide
                          └─→  STAGING_AZURESTORAGE.order_detail  ──┘─→  INTEGRATION.order_detail  ──┘
```

Vier niveaus: parquet-bestanden → Bronze → Silver (twee tabellen + MV) → Gold.

### Stap-voor-stap: Catalog Explorer

1. Ga in de Databricks-sidebar naar **Catalog**.
2. Navigeer naar `DEMO` → `DATAMART` → `sales_lines_wide`.
3. Klik op het tabblad **Lineage**.
4. Je ziet één directe upstream-node: `INTEGRATION.sales_line`.
5. Klik op `sales_line` — je ziet de twee upstream-Silver-tabellen:
   `INTEGRATION.order_header` en `INTEGRATION.order_detail`.
6. Klik nogmaals omhoog — je bereikt de Bronze-tabellen en daarna de parquet-volumes.
7. Bekijk het tabblad **Column lineage** op `sales_lines_wide` — klik op
   `order_total` en trace de waarde terug tot `ORDER_TOTAL` in de Bronze-parquet.

> **Demo talking point:** Column-level lineage over vier lagen heen — van Gold-wide-tabel
> terug tot de ruwe bronkolom in een parquet-bestand. Geen enkele ETL-annotatie vereist:
> UC registreert dit automatisch bij elke pipeline-run.

In [ ]:
# Programmatische lineage-check voor DATAMART.sales_lines_wide (upstream)
gold_wide_upstream_sql = f"""
SELECT
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM system.lineage.table_lineage
WHERE LOWER(target_table_full_name) = LOWER('{gold_sales_lines_wide}')
ORDER BY created_at DESC
LIMIT 20
"""

gold_wide_upstream_df = spark.sql(gold_wide_upstream_sql)
count = gold_wide_upstream_df.count()
print(f"Upstream-herkomst voor '{gold_sales_lines_wide}': {count} record(s)")
display(gold_wide_upstream_df)

## Sectie 6 — Programmatische lineage-query: einde-tot-einde keten

### Wat laat dit zien?

Naast de visuele Lineage viewer in Catalog Explorer is lineage ook **direct querybaar**
via SQL op de `system.lineage.table_lineage`-systeemtabel. Dit is waardevol voor:

- **Impact-analyse:** Welke downstream-tabellen worden geraakt als ik brontabel X aanpas?
- **Audit-rapportage:** Welke tabellen hebben `DATAMART.sales_lines_wide` gevoed in de
  afgelopen 30 dagen?
- **Compliance:** Toon aan dat alle Gold-tabellen terug te traceren zijn naar gecertificeerde
  bronsystemen.
- **Orchestratie:** Automatisch de juiste volgorde bepalen voor herverwerking bij
  bron-schema-wijzigingen.

### Kolommen in `system.lineage.table_lineage`

| Kolom | Beschrijving |
|---|---|
| `source_table_full_name` | Volledig gekwalificeerde naam van de brontabel (`catalog.schema.table`) |
| `target_table_full_name` | Volledig gekwalificeerde naam van de doeltabel |
| `entity_type` | Type bron: `TABLE`, `PATH` (voor bestanden/volumes) |
| `created_by` | Gebruiker of service principal die de lineage-relatie vastlegde |
| `created_at` | Tijdstip van vastleggen |

De volgende query bouwt de **volledige downstream-keten** op vanuit de Bronze
`order_header`-tabel met behulp van een recursieve CTE — traverseerbaar van
Bronze helemaal door tot Gold.

In [ ]:
# Volledige downstream-keten vanuit Bronze order_header (recursieve traversal)
# Databricks SQL ondersteunt recursieve CTE's via WITH RECURSIVE.
# Elke stap volgt één upstream-downstream-relatie.
full_chain_sql = f"""
WITH RECURSIVE lineage_chain AS (
    -- Anker: directe downstream-tabellen van Bronze order_header
    SELECT
        source_table_full_name,
        target_table_full_name,
        1                        AS depth,
        entity_type,
        created_by,
        created_at
    FROM system.lineage.table_lineage
    WHERE LOWER(source_table_full_name) = LOWER('{bronze_order_header}')

    UNION ALL

    -- Recursieve stap: volg downstream-relaties
    SELECT
        tl.source_table_full_name,
        tl.target_table_full_name,
        lc.depth + 1            AS depth,
        tl.entity_type,
        tl.created_by,
        tl.created_at
    FROM system.lineage.table_lineage tl
    INNER JOIN lineage_chain lc
        ON LOWER(tl.source_table_full_name) = LOWER(lc.target_table_full_name)
    WHERE lc.depth < 5   -- veiligheidsgrens om oneindige lussen te voorkomen
)
SELECT DISTINCT
    depth,
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM lineage_chain
ORDER BY depth, source_table_full_name, target_table_full_name
"""

chain_df = spark.sql(full_chain_sql)
count = chain_df.count()
print(f"Downstream-relaties vanuit '{bronze_order_header}': {count} record(s)")
print("(depth=1: directe downstream; depth=2: twee stappen downstream; etc.)")
display(chain_df)

In [ ]:
# Impact-analyse: welke tabellen in de demo-catalog worden geraakt als Bronze order_header wijzigt?
# Praktisch: stel je voor dat je een bronkolom wil hernoemen of een schema-wijziging wil doorvoeren.
impact_sql = f"""
SELECT DISTINCT
    target_table_full_name                                                 AS impacted_table,
    CASE
        WHEN LOWER(target_table_full_name) LIKE LOWER('%{silver_schema}%') THEN 'Silver'
        WHEN LOWER(target_table_full_name) LIKE LOWER('%{gold_schema}%')   THEN 'Gold'
        ELSE 'Overig'
    END                                                                    AS medallion_layer,
    entity_type,
    created_by
FROM system.lineage.table_lineage
WHERE LOWER(source_table_full_name) = LOWER('{bronze_order_header}')
   OR LOWER(source_table_full_name) IN (
       SELECT LOWER(target_table_full_name)
       FROM system.lineage.table_lineage
       WHERE LOWER(source_table_full_name) = LOWER('{bronze_order_header}')
   )
ORDER BY medallion_layer, impacted_table
"""

impact_df = spark.sql(impact_sql)
count = impact_df.count()
print(f"Direct en indirect gerichte tabellen bij wijziging van '{bronze_order_header}': {count}")
display(impact_df)

In [ ]:
# Upstream-trace vanuit Gold terug naar Bronze — traverseerbaar audit-spoor
# Vraag: "Van welke bronstabellen en -bestanden leest DATAMART.sales_lines_wide?"
gold_to_bronze_sql = f"""
WITH RECURSIVE upstream_chain AS (
    -- Anker: directe upstream-tabellen van de Gold wide-tabel
    SELECT
        source_table_full_name,
        target_table_full_name,
        1                        AS depth,
        entity_type,
        created_by,
        created_at
    FROM system.lineage.table_lineage
    WHERE LOWER(target_table_full_name) = LOWER('{gold_sales_lines_wide}')

    UNION ALL

    -- Recursieve stap: volg upstream-relaties
    SELECT
        tl.source_table_full_name,
        tl.target_table_full_name,
        uc.depth + 1            AS depth,
        tl.entity_type,
        tl.created_by,
        tl.created_at
    FROM system.lineage.table_lineage tl
    INNER JOIN upstream_chain uc
        ON LOWER(tl.target_table_full_name) = LOWER(uc.source_table_full_name)
    WHERE uc.depth < 5
)
SELECT DISTINCT
    depth,
    source_table_full_name,
    target_table_full_name,
    entity_type,
    created_by,
    created_at
FROM upstream_chain
ORDER BY depth, source_table_full_name
"""

gold_to_bronze_df = spark.sql(gold_to_bronze_sql)
count = gold_to_bronze_df.count()
print(f"Upstream-keten vanuit '{gold_sales_lines_wide}' terug naar Bronze: {count} record(s)")
print("(depth=1: directe upstream; hogere depth: verder terug in de keten)")
display(gold_to_bronze_df)

## Sectie 7 — Waarde van Unity Catalog Lineage

Dit notebook heeft de lineage-keten van Bronze tot Gold gedemonstreerd — zowel visueel
via Catalog Explorer als programmatisch via `system.lineage.table_lineage`.

### Vier kernwaarden voor klanten

#### 1. Impact-analyse

Voordat je een schema-wijziging doorvoert (bijv. een kolom hernoemen in
`STAGING_AZURESTORAGE.order_header`), query je `system.lineage.table_lineage` om te
zien welke downstream-tabellen worden geraakt. De query in Sectie 6 geeft je in
seconden een compleet impactrapport — inclusief Silver Streaming Tables, Materialised
Views en Gold-aggregaten.

```sql
-- Welke tabellen worden geraakt als ik STAGING_AZURESTORAGE.order_header aanpas?
SELECT DISTINCT target_table_full_name
FROM   system.lineage.table_lineage
WHERE  LOWER(source_table_full_name) = 'demo.staging_azurestorage.order_header'
```

#### 2. Debugging

Een dashboard toont een onverwachte daling in `total_revenue` op
`DATAMART.daily_sales_by_truck`. Via Lineage trace je de waarde terug:

1. `daily_sales_by_truck` ← `INTEGRATION.order_header` (Silver)
2. `order_header` (Silver) ← `STAGING_AZURESTORAGE.order_header` (Bronze)
3. `order_header` (Bronze) ← specifieke parquet-bestanden in het volume

Op elk niveau kun je de Delta-tabel inspecteren, de versiegeschiedenis raadplegen
(`DESCRIBE HISTORY`) en de relevante Workflow-run-logs bekijken.

#### 3. Compliance en governance

Regulatoire vereisten (AVG/GDPR, SOC 2, ISO 27001) vragen bewijs dat persoonsgegevens
traceerbaar zijn naar hun bron. UC Lineage levert:

- Automatisch vastgelegd, onveranderlijk herkomstspoor
- Querybaar via SQL (geen apart governance-product)
- Kolom-level detail: welke bronkolom levert de waarde voor `customer_id` in Gold?

#### 4. Automatische catalogisering

Nieuwe tabellen, views en pipelines worden automatisch opgenomen in de lineage-graph
zodra ze hun eerste schrijfoperatie uitvoeren. Geen handmatige registratie, geen
API-calls, geen aparte metadata-tool.

### Overzicht: gedemonstreerde lineage-relaties

| Van | Naar | Via |
|---|---|---|
| Parquet-volume (Azure Storage) | `STAGING_AZURESTORAGE.order_header` | Auto Loader / batch parquet-read |
| Parquet-volume (Azure Storage) | `STAGING_AZURESTORAGE.order_detail` | Auto Loader / batch parquet-read |
| `STAGING_AZURESTORAGE.order_header` | `INTEGRATION.order_header` | DLT CDF + FLOW AUTO CDC |
| `STAGING_AZURESTORAGE.order_detail` | `INTEGRATION.order_detail` | DLT CDF + FLOW AUTO CDC |
| `INTEGRATION.order_header` | `INTEGRATION.sales_line` | DLT Materialised View (JOIN) |
| `INTEGRATION.order_detail` | `INTEGRATION.sales_line` | DLT Materialised View (JOIN) |
| `INTEGRATION.order_header` | `DATAMART.daily_sales_by_truck` | DLT Materialised View (aggregaat) |
| `INTEGRATION.order_header` | `DATAMART.daily_sales_by_location` | DLT Materialised View (aggregaat) |
| `INTEGRATION.order_header` | `DATAMART.monthly_revenue_by_currency` | DLT Materialised View (aggregaat) |
| `INTEGRATION.sales_line` | `DATAMART.sales_lines_wide` | DLT Materialised View (wide-tabel) |

### Volgende stappen (voor de klant)

- Stel een Databricks SQL Alert in op lineage-wijzigingen (via de audit log) wanneer
  nieuwe downstream-tabellen worden toegevoegd aan kritieke brontabellen.
- Exporteer de lineage-query-resultaten naar een externe data catalogus (bijv. Microsoft
  Purview of Collibra) via een geplande Databricks-Job.
- Combineer met Audit Logs (zie `demo_showcase/audit_logs.ipynb`) voor een volledig
  compliance-spoor: wie heeft data verwerkt (audit) + welke tabellen zijn geraakt (lineage).
- Gebruik kolom-level lineage in Catalog Explorer om de herkomst van specifieke
  KPI-kolommen (bijv. `total_revenue`) te documenteren voor regulatoire rapportage.